In [1]:
# assign directory
import git
from pathlib import Path
import os
ROOT_DIR = Path(git.Repo('.', search_parent_directories=True).working_tree_dir)
os.chdir(os.path.join(ROOT_DIR, "utilities"))
from transform_audio import *
os.chdir(os.path.join(ROOT_DIR, "dataset-preparation"))

respiratory_dir  = os.path.join(ROOT_DIR, 'raw-data', 'respiratory')
data_dir = os.path.join(respiratory_dir, 'audio_and_txt_files')

In [2]:
# for each recording name there is a .wav and a .txt file

recording_names = [name[:-4] for name in os.listdir(data_dir) if name.endswith('.wav')]
recording_names.sort()
len(recording_names)

920

In [3]:
# filter sampling rate to 44100

def get_sr(recording_name):
    return wavfile.read(os.path.join(data_dir, recording_name) + '.wav')[0]
recording_names = [name for name in recording_names if get_sr(name) == 44100]
len(recording_names)

824

In [4]:
# generate recording metadata

metadata_dir = os.path.join(respiratory_dir, 'metadata')
Path(metadata_dir).mkdir(exist_ok=True)

recording_metadata = pd.DataFrame({'recording name': recording_names})
recording_metadata[
    ['patient number', 'recording index', 'chest location','acquisition mode', 'recording equipment']
] = recording_metadata['recording name'].str.split('_', expand=True)
recording_metadata.set_index('recording name', inplace=True)
recording_metadata.to_csv(os.path.join(metadata_dir, 'recording_metadata.csv'))
recording_metadata

,patient number,recording index,chest location,acquisition mode,recording equipment
recording name,,,,,
101_1b1_Al_sc_Meditron,101,1b1,Al,sc,Meditron
101_1b1_Pr_sc_Meditron,101,1b1,Pr,sc,Meditron
102_1b1_Ar_sc_Meditron,102,1b1,Ar,sc,Meditron
103_2b2_Ar_mc_LittC2SE,103,2b2,Ar,mc,LittC2SE
105_1b1_Tc_sc_Meditron,105,1b1,Tc,sc,Meditron
...,...,...,...,...,...
224_1b2_Al_sc_Meditron,224,1b2,Al,sc,Meditron
225_1b1_Pl_sc_Meditron,225,1b1,Pl,sc,Meditron
226_1b1_Al_sc_Meditron,226,1b1,Al,sc,Meditron


In [5]:
# generate patient metadata

demographic_info = pd.read_csv(
    os.path.join(respiratory_dir, 'demographic_info.txt'), 
    sep=' ', header=None
)
demographic_info.columns = ['patient number', 'age', 'sex', 'adult bmi', 'child weight', 'child height']

diagnoses = pd.read_csv(os.path.join(respiratory_dir, 'patient_diagnosis.csv'), header=None)
diagnoses.columns = ['patient number', 'diagnosis']

patient_metadata = demographic_info.merge(diagnoses, on='patient number')
patient_metadata.set_index('patient number', inplace=True)
patient_metadata.to_csv(os.path.join(metadata_dir, 'patient_metadata.csv'))
patient_metadata

,age,sex,adult bmi,child weight,child height,diagnosis
patient number,,,,,,
101,3.00,F,NaN,19.0,99.0,URTI
102,0.75,F,NaN,9.8,73.0,Healthy
103,70.00,F,33.00,NaN,NaN,Asthma
104,70.00,F,28.47,NaN,NaN,COPD
105,7.00,F,NaN,32.0,135.0,URTI
...,...,...,...,...,...,...
222,60.00,M,NaN,NaN,NaN,COPD
223,NaN,NaN,NaN,NaN,NaN,COPD
224,10.00,F,NaN,32.3,143.0,Healthy


In [14]:
# split data into cycles

MIN_CYCLE_LEN = 1.
MAX_CYCLE_LEN = 5.

cycles_dir = os.path.join(respiratory_dir, 'respiratory_cycles')
os.mkdir(cycles_dir)

cycle_metadata = pd.DataFrame(
    columns=['cycle name', 'recording name', 'cycle index', 
             'cycle start', 'cycle end', 'crackles', 'wheezes']
)
cycle_metadata.set_index('cycle name', inplace=True)

for i, name in enumerate(recording_names):
    rate, signal = wavfile.read(os.path.join(data_dir, name) + '.wav')
    txt_table = pd.read_table(os.path.join(data_dir, name) + '.txt', header=None)
    cycle_idx = 0
    for start, end, crackles, wheezes in txt_table.itertuples(index=False, name=None):
        if not (MIN_CYCLE_LEN <= end - start <= MAX_CYCLE_LEN):
            continue
        cycle_name = f'cycle_{i:03}_{cycle_idx:02}'
        cycle_metadata.loc[cycle_name] = [name, cycle_idx, start, end, crackles, wheezes]
        start_idx, end_idx = int(start * rate), int(end * rate)
        wavfile.write(os.path.join(cycles_dir, cycle_name + '.wav'), rate, signal[start_idx:end_idx])
        cycle_idx += 1

cycle_metadata.to_csv(os.path.join(metadata_dir, 'cycle_metadata.csv'))

## --- Old preprocessing (k cycles per recording) ---

### Split audio and text into separate directories, filter sampling rate to 44100

In [ ]:
audio_dir = os.path.join(respiratory_dir, 'audio_files')
text_dir = os.path.join(respiratory_dir, 'text_files')

if not (os.path.exists(audio_dir) and os.path.exists(text_dir)):
    os.mkdir(audio_dir)
    os.mkdir(text_dir)

    file_names = [name[:-4] for name in os.listdir(data_dir) if name.endswith('.wav')]
    for name in file_names:
        data_path = os.path.join(data_dir, name)
        if wavfile.read(data_path + '.wav')[0] != 44100:
            continue
        shutil.copy(data_path + '.wav', os.path.join(audio_dir, name) + '.wav')
        shutil.copy(data_path + '.txt', os.path.join(text_dir, name) + '.txt')

file_names = [name[:-4] for name in os.listdir(audio_dir)]
audio_file_list = [os.path.join(audio_dir, name) + '.wav' for name in file_names]
text_file_list = [os.path.join(text_dir, name) + '.txt' for name in file_names]
print(f'{len(file_names)} files remaining')

824 files remaining


### Truncate by respiratory cycle

In [ ]:
CYCLES_TO_KEEP = 2 # can be at most 2 for compatibility with all files

def trim_audio_file(input_path, start_time, end_time, output_path):
    rate, signal = wavfile.read(input_path)
    start, end = int(start_time * rate), int(end_time * rate)
    wavfile.write(output_path, rate, signal[start:end])

trimmed_audio_dir = os.path.join(respiratory_dir, 'trimmed_audio_files')
if not os.path.exists(trimmed_audio_dir):
    os.mkdir(trimmed_audio_dir)

    for name, audio, text in zip(file_names, audio_file_list, text_file_list):
        table = pd.read_table(text, header=None)
        start_time = table.iloc[0, 0]
        end_time = table.iloc[CYCLES_TO_KEEP - 1, 1]
        output_path = os.path.join(trimmed_audio_dir, name) + '.wav'
        trim_audio_file(audio, start_time, end_time, output_path)

audio_file_list = [os.path.join(trimmed_audio_dir, name) + '.wav' for name in file_names]